# EA3 - Actividad 3.3: Dashboard en Tiempo Real

## Objetivos
- Visualizar datos provenientes de un pipeline de streaming
- Crear metricas clave actualizables
- Construir graficos de monitoreo con refresco automatico

> **IMPORTANTE:** Este notebook requiere el perfil **completo** de Docker.
> ```bash
> docker-compose --profile completo up -d
> ```
> Espera ~1 minuto para que Kafka y los demas servicios inicien completamente.

## Conceptos Clave

### Visualizacion de Datos de Streaming

En un sistema de procesamiento en tiempo real, es fundamental poder **visualizar** lo que esta ocurriendo. Un dashboard de monitoreo nos permite:

- **Observar tendencias:** Ver como evolucionan las metricas a lo largo del tiempo
- **Detectar anomalias:** Identificar comportamientos inusuales rapidamente
- **Tomar decisiones:** Reaccionar en tiempo real ante cambios en los datos

### Patrones de Refresco

| Patron | Descripcion | Uso |
|--------|-------------|-----|
| **Polling** | Consultar periodicamente los datos nuevos | Dashboards simples |
| **Push** | El servidor notifica cuando hay datos nuevos | Dashboards en produccion |
| **Batch refresh** | Actualizar cada N segundos con datos acumulados | Jupyter notebooks |

En este notebook usaremos **batch refresh** con `clear_output()` de IPython para simular actualizaciones en tiempo real dentro de Jupyter.

### Metricas Clave para Transacciones

- **Total de transacciones:** Volumen de actividad
- **Monto total:** Valor monetario acumulado
- **Ticket promedio:** Monto promedio por transaccion
- **Ventas por producto:** Distribucion de la demanda
- **Ventas por tienda:** Rendimiento por punto de venta
- **Metodo de pago:** Preferencias de pago de los clientes

## Setup

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
import matplotlib.pyplot as plt
import pandas as pd
import time
import os
from IPython.display import clear_output, display

spark = SparkSession.builder \
    .appName("dashboard_rt") \
    .master("local[*]") \
    .getOrCreate()

%matplotlib inline
print(f"Spark version: {spark.version}")
print("Setup completado.")

## 1. Cargar Datos

Leemos los datos procesados por el notebook 03 (Parquet). Si no existen, generamos datos de ejemplo con el script de streaming.

In [ ]:
# Si no hay datos de streaming, generar muestra
parquet_path = "/home/jovyan/datos/streaming/output/transacciones"
if not os.path.exists(parquet_path):
    print("No se encontraron datos Parquet. Generando datos de ejemplo...")
    # Fallback: generar JSONL y leer
    %run /home/jovyan/scripts/generar_datos_streaming.py --tipo transacciones --archivo /home/jovyan/datos/streaming/transacciones_sample.jsonl --cantidad 2000
    df = spark.read.json("/home/jovyan/datos/streaming/transacciones_sample.jsonl")
else:
    print("Leyendo datos desde Parquet (generados por notebook 03)...")
    df = spark.read.parquet(parquet_path)

df.show(5)
print(f"Total registros: {df.count()}")

## 2. Calcular Metricas Principales

Extraemos las metricas clave del negocio a partir de los datos cargados.

In [ ]:
total_tx = df.count()
monto_total = df.agg(F.sum("monto")).collect()[0][0]
monto_promedio = df.agg(F.avg("monto")).collect()[0][0]

print("=" * 50)
print("       METRICAS PRINCIPALES")
print("=" * 50)
print(f"  Total transacciones:  {total_tx:,}")
print(f"  Monto total:          ${monto_total:,.0f}")
print(f"  Monto promedio:       ${monto_promedio:,.0f}")
print("=" * 50)

## 3. Grafico de Barras: Ventas por Producto

Visualizamos el total de ventas desglosado por producto.

In [ ]:
# Ventas por producto
df_prod = df.groupBy("producto").agg(
    F.sum("monto").alias("total_ventas"),
    F.count("*").alias("num_transacciones")
).orderBy("total_ventas", ascending=False).toPandas()

fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(df_prod["producto"], df_prod["total_ventas"], color="steelblue")
ax.set_xlabel("Total Ventas ($)")
ax.set_ylabel("Producto")
ax.set_title("Ventas Totales por Producto")
plt.tight_layout()
plt.show()

## 4. Grafico de Barras: Transacciones por Metodo de Pago

Analizamos la distribucion de metodos de pago utilizados por los clientes.

In [ ]:
# Transacciones por metodo de pago
df_pago = df.groupBy("metodo_pago").count().orderBy("count", ascending=False).toPandas()

fig, ax = plt.subplots(figsize=(8, 6))
colores = ["#2196F3", "#4CAF50", "#FF9800", "#F44336", "#9C27B0"]
ax.bar(df_pago["metodo_pago"], df_pago["count"], color=colores[:len(df_pago)])
ax.set_xlabel("Metodo de Pago")
ax.set_ylabel("Numero de Transacciones")
ax.set_title("Transacciones por Metodo de Pago")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

## 5. Funcion de Dashboard con Refresco

Creamos una funcion que genera un dashboard completo con 4 graficos. Usando `clear_output()` podemos simular un refresco automatico.

In [ ]:
def mostrar_dashboard(df, iteracion=None):
    """Genera un dashboard con 4 graficos a partir de un DataFrame de transacciones."""
    clear_output(wait=True)
    
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    titulo = "Dashboard de Transacciones en Tiempo Real"
    if iteracion is not None:
        titulo += f" (Actualizacion #{iteracion})"
    fig.suptitle(titulo, fontsize=16, fontweight="bold")
    
    # Grafico 1: Ventas por producto (barras horizontales)
    df_prod = df.groupBy("producto").agg(
        F.sum("monto").alias("total")
    ).toPandas()
    axes[0, 0].barh(df_prod["producto"], df_prod["total"], color="steelblue")
    axes[0, 0].set_title("Ventas por Producto")
    axes[0, 0].set_xlabel("Total ($)")
    
    # Grafico 2: Distribucion por metodo de pago (barras)
    df_pago = df.groupBy("metodo_pago").count().toPandas()
    colores = ["#2196F3", "#4CAF50", "#FF9800", "#F44336", "#9C27B0"]
    axes[0, 1].bar(df_pago["metodo_pago"], df_pago["count"],
                   color=colores[:len(df_pago)])
    axes[0, 1].set_title("Transacciones por Metodo de Pago")
    axes[0, 1].tick_params(axis="x", rotation=45)
    
    # Grafico 3: Top tiendas por ventas
    df_tienda = df.groupBy("tienda_id").agg(
        F.sum("monto").alias("total")
    ).orderBy("total", ascending=False).toPandas()
    axes[1, 0].bar(
        df_tienda["tienda_id"].astype(str),
        df_tienda["total"],
        color="coral"
    )
    axes[1, 0].set_title("Ventas por Tienda")
    axes[1, 0].set_xlabel("Tienda ID")
    axes[1, 0].set_ylabel("Total ($)")
    
    # Grafico 4: Metricas textuales
    total_tx = df.count()
    monto_total = df.agg(F.sum("monto")).collect()[0][0] or 0
    ticket_prom = df.agg(F.avg("monto")).collect()[0][0] or 0
    axes[1, 1].axis("off")
    metricas_texto = (
        f"Total Transacciones:\n{total_tx:,}\n\n"
        f"Monto Total:\n${monto_total:,.0f}\n\n"
        f"Ticket Promedio:\n${ticket_prom:,.0f}"
    )
    axes[1, 1].text(0.5, 0.5, metricas_texto,
                    transform=axes[1, 1].transAxes,
                    fontsize=16, verticalalignment="center",
                    horizontalalignment="center",
                    bbox=dict(boxstyle="round", facecolor="lightyellow", alpha=0.8))
    axes[1, 1].set_title("Metricas Clave")
    
    plt.tight_layout()
    plt.show()

print("Funcion mostrar_dashboard() definida.")

## 6. Loop de Actualizacion Automatica

Ejecutamos el dashboard con refresco automatico cada 3 segundos durante 5 iteraciones. En un escenario real, los datos cambiarian entre cada refresco.

In [ ]:
# Simular actualizacion periodica
for i in range(5):
    mostrar_dashboard(df, iteracion=i + 1)
    time.sleep(3)

print("Dashboard finalizado despues de 5 actualizaciones.")

---
## Ejercicios

Ahora es tu turno de practicar. Completa los siguientes ejercicios.

In [ ]:
# =============================================================
# EJERCICIO 1: Panel de metricas textuales
# =============================================================
# Calculamos y mostramos 3 metricas clave con formato

# 1. Total de transacciones
total = df.count()

# 2. Monto total de ventas
monto_total = df.agg(F.sum("monto")).collect()[0][0]

# 3. Ticket promedio
ticket_prom = df.agg(F.avg("monto")).collect()[0][0]

# Mostrar panel formateado
print(f"{'='*42}")
print(f"       PANEL DE METRICAS")
print(f"{'='*42}")
print(f"  Total transacciones:  {total:,}")
print(f"  Monto total:          ${monto_total:,.0f}")
print(f"  Ticket promedio:      ${ticket_prom:,.0f}")
print(f"{'='*42}")

> **Nota docente:** El patron `.agg(F.sum(...)).collect()[0][0]` es la forma estandar de extraer un valor escalar de un DataFrame de Spark. `collect()` trae todos los datos al driver (cuidado con DataFrames grandes), `[0]` accede a la primera fila y `[0]` al primer campo. Los f-strings con formato `{valor:,.0f}` son fundamentales para presentar datos financieros de forma legible. Error comun: olvidar `collect()` y tratar de usar el DataFrame como si fuera un numero.

In [ ]:
# =============================================================
# EJERCICIO 2: Grafico de ventas por tienda (top 10)
# =============================================================
# Agrupamos por tienda, calculamos total y graficamos top 10

# 1. Agregar y obtener top 10
df_tiendas = df.groupBy("tienda_id").agg(
    F.sum("monto").alias("total")
).orderBy("total", ascending=False).limit(10).toPandas()

# 2. Crear grafico de barras horizontales
fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(
    df_tiendas["tienda_id"].astype(str),
    df_tiendas["total"],
    color="coral"
)
ax.set_xlabel("Monto Total ($)")
ax.set_ylabel("Tienda ID")
ax.set_title("Top 10 Tiendas por Ventas")

# Agregar valores en las barras
for i, (val, name) in enumerate(zip(df_tiendas["total"], df_tiendas["tienda_id"])):
    ax.text(val, i, f"  ${val:,.0f}", va='center', fontsize=9)

plt.tight_layout()
plt.show()

> **Nota docente:** El flujo Spark DataFrame -> `.toPandas()` -> matplotlib es el patron estandar para visualizacion. `.toPandas()` convierte el DataFrame distribuido de Spark a un pandas DataFrame local (solo viable cuando el resultado es pequeno, como un top 10). Error comun: llamar `.toPandas()` sobre un DataFrame de millones de filas, lo que saturaria la memoria del driver. El `.astype(str)` en `tienda_id` es necesario porque matplotlib interpreta enteros como posiciones numericas en el eje. Los valores en las barras mejoran la legibilidad del grafico.

---
## Resumen

En esta actividad aprendimos:

1. **Visualizacion de streaming:** Importancia de observar datos en tiempo real para monitoreo y toma de decisiones
2. **Patrones de refresco:** Polling, push y batch refresh para actualizar dashboards
3. **Metricas clave:** Total transacciones, monto total y ticket promedio como indicadores principales
4. **Graficos con matplotlib:** Barras horizontales, barras verticales y paneles de metricas
5. **Refresco automatico:** Uso de `clear_output()` para simular actualizaciones en Jupyter
6. **Dashboard integrado:** Funcion reutilizable con 4 graficos en una sola vista

---
## Desafio Extra (Opcional)

**Dashboard completo con 4 graficos y refresco automatico:**

Crea un dashboard con los siguientes 4 graficos y configuralo para que se refresque automaticamente cada 5 segundos durante 30 segundos (6 iteraciones):

1. Barras horizontales: ventas por producto
2. Grafico de torta (pie): distribucion por metodo de pago
3. Grafico de linea temporal: monto acumulado por timestamp
4. Barras: top 5 tiendas por volumen de transacciones

In [ ]:
# =============================================================
# DESAFIO: Dashboard completo con refresco automatico
# =============================================================
# Funcion que genera 4 graficos en una grilla 2x2

def dashboard_completo(df, iteracion=None):
    """Dashboard con 4 graficos: barras producto, torta pagos, linea temporal, top tiendas."""
    clear_output(wait=True)
    
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    titulo = "Dashboard Completo de Transacciones"
    if iteracion is not None:
        titulo += f" (Actualizacion #{iteracion})"
    fig.suptitle(titulo, fontsize=16, fontweight="bold")
    
    # [0,0] Barras horizontales: ventas totales por producto
    df_prod = df.groupBy("producto").agg(
        F.sum("monto").alias("total")
    ).orderBy("total", ascending=False).toPandas()
    axes[0, 0].barh(df_prod["producto"], df_prod["total"], color="steelblue")
    axes[0, 0].set_title("Ventas por Producto")
    axes[0, 0].set_xlabel("Total ($)")
    
    # [0,1] Grafico de torta: distribucion metodo de pago
    df_pago = df.groupBy("metodo_pago").count().toPandas()
    colores_pie = ["#2196F3", "#4CAF50", "#FF9800", "#F44336", "#9C27B0"]
    axes[0, 1].pie(
        df_pago["count"],
        labels=df_pago["metodo_pago"],
        autopct='%1.1f%%',
        colors=colores_pie[:len(df_pago)],
        startangle=90
    )
    axes[0, 1].set_title("Distribucion por Metodo de Pago")
    
    # [1,0] Linea temporal: ventas acumuladas por minuto
    df_temporal = df.withColumn(
        "minuto", F.date_trunc("minute", F.to_timestamp("timestamp"))
    ).groupBy("minuto").agg(
        F.sum("monto").alias("total")
    ).orderBy("minuto").toPandas()
    
    if not df_temporal.empty:
        axes[1, 0].plot(df_temporal["minuto"], df_temporal["total"].cumsum(),
                        color="green", linewidth=2, marker='o', markersize=3)
        axes[1, 0].set_title("Ventas Acumuladas en el Tiempo")
        axes[1, 0].set_xlabel("Tiempo")
        axes[1, 0].set_ylabel("Monto Acumulado ($)")
        axes[1, 0].tick_params(axis='x', rotation=45)
    
    # [1,1] Barras: top 5 tiendas por numero de transacciones
    df_top = df.groupBy("tienda_id").count()         .orderBy("count", ascending=False).limit(5).toPandas()
    axes[1, 1].bar(
        df_top["tienda_id"].astype(str),
        df_top["count"],
        color="coral"
    )
    axes[1, 1].set_title("Top 5 Tiendas por Volumen")
    axes[1, 1].set_xlabel("Tienda ID")
    axes[1, 1].set_ylabel("Num Transacciones")
    
    plt.tight_layout()
    plt.show()

print("Funcion dashboard_completo() definida.")

> **Nota docente:** Este dashboard combina cuatro tipos de visualizacion diferentes, cada uno apropiado para su metrica: (1) barras horizontales para comparar categorias con nombres largos, (2) torta/pie para proporciones de un total, (3) linea para tendencias temporales, y (4) barras verticales para ranking. El uso de `date_trunc("minute", ...)` agrupa timestamps al minuto mas cercano, y `.cumsum()` de pandas calcula la suma acumulada. Error comun: no usar `clear_output(wait=True)` antes de dibujar, lo que causa graficos duplicados en el refresco.

In [ ]:
# Ejecutar dashboard con refresco automatico: 6 iteraciones, cada 5 segundos
for i in range(6):
    dashboard_completo(df, iteracion=i + 1)
    time.sleep(5)

print("Dashboard finalizado despues de 6 actualizaciones (30 segundos).")

> **Nota docente:** En este loop, los datos no cambian entre iteraciones porque estamos leyendo un archivo estatico. En un escenario real, el DataFrame se recargaria en cada iteracion con datos frescos (ej: leyendo nuevamente desde Parquet o consultando una tabla que se actualiza via streaming). Este patron demuestra la mecanica del refresco; en produccion se usarian herramientas como Grafana, Superset o Streamlit para dashboards en tiempo real.

In [ ]:
# Detener SparkSession
spark.stop()
print("SparkSession detenida correctamente.")